# Vision Transformer (ViT) Animal Classifier

This notebook demonstrates animal classification using **Vision Transformers (ViT)** — specifically `google/vit-base-patch16-224` pretrained on ImageNet-21k and fine-tuned on ImageNet-1k.

## Architecture Overview
ViT divides an image into fixed-size patches and processes them as a sequence using a standard Transformer encoder:

1. **Patch Embedding** — Split 224×224 image into 16×16 patches → 196 patch tokens
2. **[CLS] Token** — Learnable classification token prepended to the sequence
3. **Positional Embeddings** — Learnable 1D position encodings added to each token
4. **Transformer Encoder** — 12 layers of Multi-Head Self-Attention + MLP blocks
5. **Classification Head** — Linear layer on the [CLS] token output

**Key difference from CNN:** ViT has no inductive biases (no local connectivity, no translation equivariance). It learns global relationships from the first layer.

In [ ]:
# Install dependencies if needed
# !pip install transformers torch Pillow numpy matplotlib requests

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import urllib.request
import json
import io
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor

print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Pretrained ViT-B/16

In [ ]:
MODEL_ID = 'google/vit-base-patch16-224'

print(f'Loading model: {MODEL_ID}')
processor = ViTImageProcessor.from_pretrained(MODEL_ID)
vit_model = ViTForImageClassification.from_pretrained(MODEL_ID)
vit_model = vit_model.to(device)
vit_model.eval()

total_params = sum(p.numel() for p in vit_model.parameters())
print(f'\nModel loaded successfully!')
print(f'Total parameters: {total_params:,}')
print(f'Model size (approx): {total_params * 4 / 1e6:.1f} MB (float32)')
print(f'Number of output classes: {vit_model.config.num_labels}')

## 2. ViT Architecture Deep Dive

In [ ]:
cfg = vit_model.config
print('ViT-B/16 Configuration:')
print('=' * 50)
print(f'Image size:            {cfg.image_size} × {cfg.image_size}')
print(f'Patch size:            {cfg.patch_size} × {cfg.patch_size}')
num_patches = (cfg.image_size // cfg.patch_size) ** 2
print(f'Number of patches:     {num_patches} ({cfg.image_size//cfg.patch_size}×{cfg.image_size//cfg.patch_size} grid)')
print(f'Sequence length:       {num_patches + 1} (patches + [CLS] token)')
print(f'Hidden size (d_model): {cfg.hidden_size}')
print(f'Num attention heads:   {cfg.num_attention_heads}')
print(f'Head dimension:        {cfg.hidden_size // cfg.num_attention_heads}')
print(f'Num transformer layers:{cfg.num_hidden_layers}')
print(f'MLP intermediate size: {cfg.intermediate_size}')
print(f'Output classes:        {cfg.num_labels}')

print('\nTop-level modules:')
print('-' * 50)
for name, module in vit_model.named_children():
    num_p = sum(p.numel() for p in module.parameters())
    print(f'{name:20s} | params: {num_p:>10,}')

## 3. Preprocessing Pipeline

In [ ]:
def preprocess_image_vit(img: Image.Image) -> dict:
    """Preprocess a PIL image using the ViT processor."""
    if img.mode != 'RGB':
        img = img.convert('RGB')
    inputs = processor(images=img, return_tensors='pt')
    return {k: v.to(device) for k, v in inputs.items()}

print('Processor config:')
print(f'  Image size:  {processor.size}')
print(f'  Image mean:  {processor.image_mean}')
print(f'  Image std:   {processor.image_std}')
print(f'  Resample:    {processor.resample}')

## 4. Load Labels & Define Animal Classes

In [ ]:
# ViT model has built-in id2label mapping
id2label = vit_model.config.id2label
print(f'Labels loaded: {len(id2label)} classes')

# Animal class indices (same ImageNet-1k as CNN for fair comparison)
ANIMAL_CLASS_RANGES = [
    (0, 397),
    (407, 411),
    (665, 668),
]

ANIMAL_INDICES = set()
for start, end in ANIMAL_CLASS_RANGES:
    ANIMAL_INDICES.update(range(start, end + 1))

print(f'Animal classes: {len(ANIMAL_INDICES)}')
print(f'Sample labels: {[id2label[i] for i in list(ANIMAL_INDICES)[:8]]}')

## 5. Inference Function

In [ ]:
def predict_vit(image: Image.Image, top_k: int = 5) -> dict:
    """
    Run ViT inference on a PIL image.
    
    Returns:
        dict with keys:
          - 'top_predictions': list of (label, probability) tuples
          - 'all_probabilities': numpy array of 1000 class probs
          - 'predicted_class': top-1 predicted label
          - 'confidence': top-1 confidence score
          - 'is_animal': whether top prediction is an animal class
    """
    inputs = preprocess_image_vit(image)

    with torch.no_grad():
        outputs = vit_model(**inputs)
        logits  = outputs.logits                          # (1, 1000)
        probs   = torch.softmax(logits, dim=-1)           # (1, 1000)
        probs_np = probs.squeeze().cpu().numpy()          # (1000,)

    top_indices = probs_np.argsort()[::-1][:top_k]
    top_predictions = [(id2label[i], float(probs_np[i])) for i in top_indices]

    top1_idx = top_indices[0]
    return {
        'top_predictions': top_predictions,
        'all_probabilities': probs_np,
        'predicted_class': id2label[top1_idx],
        'confidence': float(probs_np[top1_idx]),
        'is_animal': top1_idx in ANIMAL_INDICES
    }

print('predict_vit() function defined.')

## 6. Test on a Sample Animal Image

In [ ]:
SAMPLE_IMAGE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Walking_tiger_female.jpg/320px-Walking_tiger_female.jpg'

try:
    with urllib.request.urlopen(SAMPLE_IMAGE_URL) as response:
        img_data = response.read()
    sample_image = Image.open(io.BytesIO(img_data)).convert('RGB')
    print(f'Sample image loaded: {sample_image.size} pixels')
except Exception as e:
    print(f'Could not download sample image: {e}')
    print('Please provide your own image: sample_image = Image.open("path/to/animal.jpg")')
    sample_image = None

In [ ]:
if sample_image is not None:
    result = predict_vit(sample_image, top_k=10)

    print(f'Predicted class: {result["predicted_class"]}')
    print(f'Confidence:      {result["confidence"]*100:.2f}%')
    print(f'Is animal class: {result["is_animal"]}')
    print()
    print('Top-10 Predictions:')
    print('-' * 40)
    for rank, (label, prob) in enumerate(result['top_predictions'], 1):
        bar = '█' * int(prob * 50)
        print(f'{rank:2d}. {label:30s} {prob*100:6.2f}% {bar}')

## 7. Visualize Predictions

In [ ]:
if sample_image is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(sample_image)
    axes[0].set_title('Input Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    labels_list = [p[0] for p in result['top_predictions']]
    probs_list  = [p[1] * 100 for p in result['top_predictions']]
    colors = ['#4CAF50' if i == 0 else '#A5D6A7' for i in range(len(labels_list))]

    bars = axes[1].barh(labels_list[::-1], probs_list[::-1], color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)', fontsize=12)
    axes[1].set_title('ViT (vit-base-patch16-224) — Top-10 Predictions', fontsize=13, fontweight='bold')
    axes[1].set_xlim(0, max(probs_list) * 1.2)

    for bar, prob in zip(bars[::-1], probs_list):
        axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                     f'{prob:.2f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('../models/vit_sample_output.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to models/vit_sample_output.png')

## 8. Attention Map Visualization (ViT Interpretability)

In [ ]:
def get_attention_maps(model, image: Image.Image, layer_idx: int = 11):
    """
    Extract attention weights from a specific transformer layer.
    Returns averaged attention from [CLS] token to all patch tokens.
    """
    inputs = preprocess_image_vit(image)

    with torch.no_grad():
        outputs = model.vit(
            **inputs,
            output_attentions=True,
            return_dict=True
        )

    # attentions: tuple of (batch, heads, seq_len, seq_len) per layer
    attn = outputs.attentions[layer_idx]   # (1, 12, 197, 197)
    attn = attn.squeeze(0)                 # (12, 197, 197)
    
    # Attention from [CLS] (index 0) to all patch tokens (indices 1:)
    cls_attn = attn[:, 0, 1:]              # (12, 196) — 12 heads
    
    # Average across heads
    avg_attn = cls_attn.mean(dim=0)        # (196,)
    avg_attn = avg_attn.cpu().numpy()
    
    # Reshape to 14×14 patch grid
    grid_size = int(avg_attn.shape[0] ** 0.5)
    attn_map = avg_attn.reshape(grid_size, grid_size)
    
    # Normalize
    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
    
    return attn_map, outputs.attentions


if sample_image is not None:
    import torch.nn.functional as F
    
    attn_map, all_attentions = get_attention_maps(vit_model, sample_image, layer_idx=11)
    print(f'Attention map shape: {attn_map.shape} (14×14 patch grid)')

    # Upsample to 224×224
    attn_upsampled = F.interpolate(
        torch.tensor(attn_map).unsqueeze(0).unsqueeze(0),
        size=(224, 224), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    img_resized = np.array(sample_image.resize((224, 224)))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('ViT Attention Map — Layer 12 (Last Transformer Block)', fontsize=13, fontweight='bold')

    axes[0].imshow(img_resized)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(attn_upsampled, cmap='hot')
    axes[1].set_title('[CLS] Attention Heatmap', fontsize=12)
    axes[1].axis('off')

    overlay = img_resized.astype(float) / 255.0
    heatmap = plt.cm.hot(attn_upsampled)[:, :, :3]
    blended = 0.55 * overlay + 0.45 * heatmap
    blended = np.clip(blended, 0, 1)

    axes[2].imshow(blended)
    axes[2].set_title('Attention Overlay', fontsize=12)
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig('../models/vit_attention_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Attention map saved to models/vit_attention_map.png')

## 9. Multi-Head Attention — Per-Head Visualization

In [ ]:
if sample_image is not None:
    import torch.nn.functional as F

    # Get all 12 heads from last layer
    attn_last = all_attentions[-1].squeeze(0)  # (12, 197, 197)
    cls_attn_all_heads = attn_last[:, 0, 1:].cpu().numpy()  # (12, 196)
    grid_size = 14

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle('ViT-B/16 — All 12 Attention Heads (Last Layer)', fontsize=14, fontweight='bold')

    for head_idx, ax in enumerate(axes.flat):
        head_attn = cls_attn_all_heads[head_idx].reshape(grid_size, grid_size)
        head_attn = (head_attn - head_attn.min()) / (head_attn.max() - head_attn.min() + 1e-8)

        head_up = F.interpolate(
            torch.tensor(head_attn).unsqueeze(0).unsqueeze(0),
            size=(224, 224), mode='bilinear', align_corners=False
        ).squeeze().numpy()

        heatmap = plt.cm.viridis(head_up)[:, :, :3]
        overlay_img = img_resized.astype(float) / 255.0
        blended_head = np.clip(0.6 * overlay_img + 0.4 * heatmap, 0, 1)

        ax.imshow(blended_head)
        ax.set_title(f'Head {head_idx + 1}', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('../models/vit_all_heads.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('All-heads attention plot saved to models/vit_all_heads.png')

## 10. Save ViT Config for Streamlit App

In [ ]:
vit_config_export = {
    'model_id': MODEL_ID,
    'image_size': cfg.image_size,
    'patch_size': cfg.patch_size,
    'num_patches': num_patches,
    'hidden_size': cfg.hidden_size,
    'num_heads': cfg.num_attention_heads,
    'num_layers': cfg.num_hidden_layers,
    'num_classes': cfg.num_labels,
    'total_params': total_params,
    'mean': processor.image_mean,
    'std': processor.image_std,
}

with open('../models/vit_config.json', 'w') as f:
    json.dump(vit_config_export, f, indent=2)

print('ViT config saved to models/vit_config.json')
print(json.dumps(vit_config_export, indent=2))

## Summary

| Property | Value |
|---|---|
| Model | ViT-B/16 |
| Parameters | ~86M |
| Patch size | 16 × 16 pixels |
| Sequence length | 197 tokens (196 patches + [CLS]) |
| Transformer layers | 12 |
| Attention heads | 12 |
| Hidden dimension | 768 |
| Pretraining | ImageNet-21k → ImageNet-1k fine-tuned |
| Top-1 accuracy | ~81.8% on ImageNet val set |

**Key ViT differences from CNN:**
- **Global receptive field** from layer 1 (every patch attends to every other)
- **No translation equivariance** — position learned via embeddings
- **Larger parameter count** — requires more data / pretraining
- **Attention interpretability** — can visualize which patches the model attends to